# 2a Escola de IA e Computação do CBPF (17-21/Ago/2026)

**Astrofísica e Cosmologia computacional**

**Aulas 2.b e 2.c - Medida de redshifts fotométricos e construção de distribuições populacionais de redshifts**

## Sumário

- Exercício 1: Redshifts fotométricos com florestas aleatórias usando magnitudes
- Exercício 2: Métricas de qualidade para redshifts fotométricos
- Exercício 3: Escolha de 'features' e comparação de casos
- Exercício 4: Construção de distribuições populacionais de redshifts fotométricos

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.metrics import accuracy_score, mean_absolute_error, mean_squared_error

from sklearn.ensemble import RandomForestRegressor
from sklearn.base import clone

## Exercício 1: Redshifts fotométricos com florestas aleatórias usando magnitudes

Neste exercício, vamos treinar um estimador de redshifts fotométricos. Usaremos uma floresta aleatória ("random forest") que aprende, a partir de uma amostra com redshifts espectroscópicos conhecidos, a relação entre as magnitudes $grizY$ do DES DR2 e o redshift da galáxia. Ao final, avaliamos o desempenho visualmente com um gráfico de $z_{\rm spec}$ vs $z_{\rm phot}$.

### 1.a) Leia, inspecione (rapidamente) os dados e crie dataframes separados com magnitudes ("features") e redshifts espectroscópicos ("target")

In [ ]:
# Load the first CSV file (assumes it contains the test data)
data = pd.read_parquet("../../data/des_dr2/des_dr2_pz_training_set_clean_vvds_vipers_gama.pq")

# The 'z' columns is the target and the 'mag_*' and 'mag_err_' columns are the predictor
redshifts = data['z']
mags = data[['mag_auto_g_dered',
             'mag_auto_r_dered',
             'mag_auto_i_dered',
             'mag_auto_z_dered',
             'mag_auto_y_dered']]

data.info()

### 1.b) Crie conjuntos de treino e teste, inicialize uma Floresta Aleatória e treine e avalie o modelo

In [ ]:
# Split the data into training and test sets (e.g., 70/30 split)
Mags_train, Mags_test, z_spec_train, z_spec_test = train_test_split(mags, redshifts, test_size=0.3, random_state=42)

# Initialize the Random Forest Regressor with hyperparameters
rf = RandomForestRegressor(n_estimators=50,
                           max_depth=30,
                           max_features=None)

# Train and evaluate the Random Forest model
rf.fit(Mags_train, z_spec_train)
z_phot_mags = rf.predict(Mags_test)
print(f"Random Forest Validation MAE: {mean_absolute_error(z_spec_test, z_phot_mags)}")

### 1.c) Faça um gráfico de $z_{\rm spec}$ vs $z_{\rm phot}$ para avaliar visualmente a performance do algoritmo

In [ ]:
def plot_zphot_zspec_comparison(z_spec,
                                z_phot,
                                ax=None,
                                outlier_thresh=0.15,
                                figsize=(5, 4.2),
                                title="",
                                gridsize=40,
                                cmap="inferno",
                                zlim=(0, 2),
                                vmax=None,
                                mincnt=1):

    # Transform to numpy arrays
    z_spec = np.asarray(z_spec, dtype=float)
    z_phot = np.asarray(z_phot, dtype=float)

    # Separate outliers from inliers
    outlier_mask = ( np.abs(z_phot - z_spec) / (1 + z_spec) ) > outlier_thresh

    # Draw into the ax we were given (one panel of a bigger figure), or make
    # our own single-panel figure; figsize only applies in the second case
    own_figure = ax is None
    if own_figure:
        fig, ax = plt.subplots(figsize=figsize)
    else:
        fig = ax.figure

    # Create hexbin plot for inliers
    z_spec_inliers = z_spec[~outlier_mask]
    z_phot_inliers = z_phot[~outlier_mask]

    hexes = None
    if (~outlier_mask).any():
        hexes = ax.hexbin(z_spec_inliers, z_phot_inliers,
                          gridsize=gridsize,
                          cmap=cmap,
                          linewidths=0,
                          mincnt=mincnt,
                          norm=LogNorm(vmin=mincnt, vmax=vmax),
                          extent=(zlim[0], zlim[1], zlim[0], zlim[1]))

    # Create scatter plot for outliers
    z_spec_outliers = z_spec[outlier_mask]
    z_phot_outliers = z_phot[outlier_mask]

    if outlier_mask.any():
        ax.scatter(z_spec_outliers, z_phot_outliers,
                   s=8,
                   c="red",
                   edgecolor='white',
                   linewidth=0.4,
                   zorder=3,
                   label='Outliers (%.2f%%)' % (outlier_mask.sum() / len(outlier_mask) * 100))
        ax.legend(loc='upper left', fontsize=8, framealpha=0.9)

    # Identity and outlier boundaries in neutral gray
    x = np.linspace(zlim[0], zlim[1], 200)
    ax.plot(x, x, color='0.25', linewidth=1.2, zorder=2)
    ax.plot(x, x + outlier_thresh * (1 + x), linestyle='--', color='0.45', linewidth=1, zorder=2)
    ax.plot(x, x - outlier_thresh * (1 + x), linestyle='--', color='0.45', linewidth=1, zorder=2)

    # Extra formatting
    if own_figure and hexes is not None:
        # In a multi-panel grid the caller adds one shared colorbar instead,
        # so that the same darkness means the same count in every panel
        fig.colorbar(hexes, ax=ax, label='Galaxies per cell')

    ax.set_xlabel('True Redshift')
    ax.set_ylabel('Predicted Redshift')
    ax.set_title(title)
    ax.set_xlim(zlim)
    ax.set_ylim(zlim)
    ax.set_aspect('equal')
    ax.grid(alpha=0.25, linewidth=0.5)
    ax.set_axisbelow(True)  # otherwise the grid is drawn on top of the hexagons

    return fig, ax, hexes


In [ ]:
_ = plot_zphot_zspec_comparison(z_spec_test, z_phot_mags, title="Random Forest Predictions on Test Set", cmap='inferno')

## Exercício 2: Métricas de qualidade para redshifts fotométricos

Para medir quantitativamente a performance do algoritmo treinado, vamos implementar as métricas-padrão da literatura de redshifts fotométricos — viés, $\sigma_{68}$, $\sigma_{\rm NMAD}$ e fração de outliers $\eta$, todas construídas sobre o resíduo normalizado $\Delta z / (1+z)$. O objetivo é calcular estas métricas em faixas de redshift e visualizá-las para entender onde a estimativa é boa e onde ela falha.

### 2.a) Defina uma função para calcular as métricas de erro e retorná-las em um dicionário

In [ ]:
def calculate_metrics(z_spec, z_phot, outlier_thresh=0.15, verbose=True):

    z_spec = np.asarray(z_spec, dtype=float)
    z_phot = np.asarray(z_phot, dtype=float)

    dz = (z_phot - z_spec) / (1 + z_spec)

    metrics = {}
    metrics['n'] = len(dz)

    # Plain errors, on the raw residuals, in units of redshift
    metrics['mae'] = mean_absolute_error(z_spec, z_phot)
    metrics['rmse'] = np.sqrt(mean_squared_error(z_spec, z_phot))

    # Three measures of the scatter of dz, in increasing order of robustness.
    # sigma_NMAD is taken around the *median* of dz, so it measures the scatter
    # with the systematic offset already removed (Ilbert et al. 2006); the 1.4826
    # rescales it to match sigma for a Gaussian.
    metrics['std'] = np.std(dz, ddof=1)                               # any outlier moves it
    metrics['nmad'] = 1.4826 * np.median(np.abs(dz - np.median(dz)))  # robust
    metrics['sigma68'] = np.quantile(np.abs(dz), 0.68)                # half-width of the central 68%

    # Systematic offset, and the uncertainty on that offset
    metrics['bias'] = np.mean(dz)
    metrics['sigbias'] = metrics['std'] / np.sqrt(metrics['n'])

    # Catastrophic outliers
    metrics['eta'] = np.mean(np.abs(dz) > outlier_thresh)
    metrics['out2'] = np.mean(np.abs(dz - metrics['bias']) > 2 * metrics['std'])
    metrics['out3'] = np.mean(np.abs(dz - metrics['bias']) > 3 * metrics['std'])

    if verbose:
        print('N galaxies:                    %6d' % metrics['n'])
        print('MAE / RMSE:                    %6.4f / %6.4f' % (metrics['mae'], metrics['rmse']))
        print('Standard deviation:            %6.4f' % metrics['std'])
        print('Normalized MAD:                %6.4f' % metrics['nmad'])
        print('Sigma_68:                      %6.4f' % metrics['sigma68'])
        print('Catastrophic outlier fraction: %6.2f %%' % (100 * metrics['eta']))
        print('Outliers beyond 2 / 3 sigma:   %6.2f %% / %6.2f %%'
              % (100 * metrics['out2'], 100 * metrics['out3']))
        print('Mean offset:                   %6.3f +/- %6.3f'
              % (metrics['bias'], metrics['sigbias']))

    return metrics

### 2.b) Defina uma função para calcular as métricas em bins de redshift e outra para fazer gráficos destas métricas

In [ ]:
# Axis label and reference line for each metric that can become a panel. The
# keys are the ones returned by `calculate_metrics`, so a metric added there
# becomes plottable here just by adding its label.
PANEL_SPECS = {
    'bias':    (r'$\langle \Delta z/(1+z) \rangle$', 0.0),
    'sigma68': (r'$\sigma_{68}$',                    0.08),
    'nmad':    (r'$\sigma_\mathrm{NMAD}$',           0.08),
    'std':     (r'$\sigma$',                         None),
    'eta':     (r'$\eta$',                           0.05),
    'out2':    (r'out$_{2\sigma}$',                  0.05),
    'out3':    (r'out$_{3\sigma}$',                  0.05),
    'mae':     ('MAE',                               None),
    'rmse':    ('RMSE',                              None),
    'n':       ('galaxies per bin',                  None),
}

DEFAULT_BINS = np.arange(0, 1.4, 0.1)


def binned_metrics(z_spec, z_phot, bins=None, bin_by='phot', min_count=25, **kwargs):

    z_spec = np.asarray(z_spec, dtype=float)
    z_phot = np.asarray(z_phot, dtype=float)
    bins = np.asarray(DEFAULT_BINS if bins is None else bins, dtype=float)

    binning_variable = z_phot if bin_by == 'phot' else z_spec

    centres = []
    stacked = {}
    for low, high in zip(bins[:-1], bins[1:]):
        in_bin = (binning_variable >= low) & (binning_variable < high)
        if in_bin.sum() < min_count:
            continue

        metrics = calculate_metrics(z_spec[in_bin], z_phot[in_bin],
                                    verbose=False, **kwargs)
 
        centres.append(0.5 * (low + high))
        for key, value in metrics.items():
            stacked.setdefault(key, []).append(value)

    if not centres:
        # Nothing survived the min_count cut; empty arrays keep the caller working
        stacked = {key: [] for key in list(PANEL_SPECS) + ['sigbias']}

    stacked = {key: np.asarray(value, dtype=float) for key, value in stacked.items()}
    stacked['z'] = np.asarray(centres, dtype=float)
    return stacked


def plot_metrics(z_spec, z_phot,
                 bins=None,
                 panels=('bias', 'sigma68', 'out2', 'out3'),
                 bin_by='phot',
                 min_count=25,
                 axes=None,
                 label=None,
                 title=None,
                 figsize=None,
                 color='#2a78d6',
                 marker='o',
                 format_axes=True,
                 path_to_save='',
                 **kwargs):


    unknown = [key for key in panels if key not in PANEL_SPECS]
    if unknown:
        raise KeyError('Unknown metric(s) %s. Available: %s'
                       % (unknown, sorted(PANEL_SPECS)))

    stats = binned_metrics(z_spec, z_phot, bins=bins, bin_by=bin_by,
                           min_count=min_count, **kwargs)

    # Draw into the axes we were given, or make our own figure
    own_figure = axes is None
    if own_figure:
        if figsize is None:
            figsize = (7.5, 1.75 * len(panels) + 0.9)
        _, axes = plt.subplots(len(panels), 1, figsize=figsize, sharex=True,
                               constrained_layout=True)
        
    axes = np.atleast_1d(axes)
    fig = axes.flat[0].figure

    style = dict(color=color, marker=marker, markersize=5.5,
                 markeredgecolor='white', markeredgewidth=0.6,
                 linewidth=1.8, label=label)

    for ax, key in zip(axes.flat, panels):
        ylabel, reference = PANEL_SPECS[key]

        if key == 'bias':
            # Only the bias panel gets error bars: it is the one place where
            # "is this consistent with zero?" is the actual question
            ax.errorbar(stats['z'], stats['bias'], yerr=stats['sigbias'],
                        elinewidth=1, capsize=0, **style)
        else:
            ax.plot(stats['z'], stats[key], **style)

        if format_axes:
            if reference is not None:
                ax.axhline(reference, color='0.55', linestyle='--', linewidth=1,
                           zorder=1)
            ax.set_ylabel(ylabel, fontsize=13)
            ax.grid(alpha=0.25, linewidth=0.5)
            ax.set_axisbelow(True)
            ax.spines[['top', 'right']].set_visible(False)  # recessive axes
            ax.tick_params(labelsize=10)

    if own_figure:
        # The x axis is whatever we binned on, and those are not the same thing:
        # binning on z_phot is the honest choice, binning on z_spec is diagnostic
        axes.flat[-1].set_xlabel(r'$z_\mathrm{phot}$' if bin_by == 'phot'
                                 else r'$z_\mathrm{spec}$', fontsize=13)
        axes.flat[0].set_xlim(DEFAULT_BINS[0] if bins is None else bins[0],
                              DEFAULT_BINS[-1] if bins is None else bins[-1])
        if title:
            fig.suptitle(title, fontsize=13)
        if label:
            axes.flat[0].legend(fontsize=10, frameon=False)

    if path_to_save:
        fig.savefig(path_to_save, dpi=150, bbox_inches='tight')

    return fig, axes

### 2.c) Calculate, plot and inspect the metrics for the photo-zs estimated in Exercise 1

In [ ]:
# As métricas do Exercício 1, agora em função do redshift em vez de um número só
_ = plot_metrics(z_spec_test, z_phot_mags,
                 title='Random Forest com magnitudes: qualidade vs redshift')

# `panels` escolhe as métricas, com as mesmas chaves devolvidas por calculate_metrics
_ = plot_metrics(z_spec_test, z_phot_mags,
                 panels=('bias','nmad', 'eta'),
                 bins=np.arange(0, 1.6, 0.15),
                 title='Mesma estimativa, outras métricas e bins mais largos')

## Exercício 3: Escolha de 'features' e comparação de casos

A qualidade dos photo-zs depende não só do algoritmo, mas também das 'features' que ele recebe. Neste exercício treinaremos a mesma floresta aleatória com diferentes conjuntos de atributos — magnitudes, cores, com e sem erros fotométricos — e compararemos a sua performance, adaptando o gráfico do Exercício 1 e as métricas do Exercício 2.

### 3.a) Crie subconjuntos dos dados com as colunas necessárias para cada um dos testes planejados

In [ ]:
redshifts = data['z']

mags = data[['mag_auto_g_dered',
             'mag_auto_r_dered',
             'mag_auto_i_dered',
             'mag_auto_z_dered',
             'mag_auto_y_dered']]

colors = data[['g_minus_r',
               'r_minus_i',
               'i_minus_z',
               'z_minus_y']]

clrmag = data[['g_minus_r',
               'r_minus_i',
               'i_minus_z',
               'z_minus_y',
               'mag_auto_i_dered']]

mags_werr = data[['mag_auto_g_dered',
                  'mag_auto_r_dered',
                  'mag_auto_i_dered',
                  'mag_auto_z_dered',
                  'mag_auto_y_dered',
                  'magerr_auto_g',
                  'magerr_auto_r',
                  'magerr_auto_i',
                  'magerr_auto_z',
                  'magerr_auto_y']]

clrmag_werr = data[['g_minus_r',
                    'r_minus_i',
                    'i_minus_z',
                    'z_minus_y',
                    'mag_auto_i_dered',
                    'magerr_auto_g',
                    'magerr_auto_r',
                    'magerr_auto_i',
                    'magerr_auto_z',
                    'magerr_auto_y']]

### 3.b) Adapte código do item **1.b** para rodar múltiplos cenários automaticamente

In [ ]:
feature_sets = {
    'Magnitudes':              mags,
    'Colors':                  colors,
    'Colors + mag_i':          clrmag,
    'Magnitudes + errors':     mags_werr,
    'Colors + mag_i + errors': clrmag_werr,
}

rf_base = RandomForestRegressor(n_estimators=50,
                                max_depth=30,
                                max_features=None)

results = {}
for name, X in feature_sets.items():
    X_train_fs, X_test_fs, y_train_fs, y_test_fs = train_test_split(X,
                                                                    redshifts,
                                                                    test_size=0.3,
                                                                    random_state=42)

    model = clone(rf_base)          # fresh, unfitted, same hyperparameters
    model.fit(X_train_fs, y_train_fs)
    y_pred_fs = model.predict(X_test_fs)

    print('\n--- %s (%d features) ---' % (name, X.shape[1]))
    results[name] = {'y_true':  y_test_fs,
                     'y_pred':  y_pred_fs,
                     'model':   model,     # kept so feature importances can be
                                           # inspected later without retraining
                     'metrics': calculate_metrics(y_test_fs, y_pred_fs)}

### 3.c) Adapte código do item **1.c** para criar um painel com todos os scatter plots

In [ ]:
# Several estimation cases on ONE figure: build the grid up front, hand each
# case its own ax, and only render/save the figure once at the end
ncols = 3
nrows = int(np.ceil(len(results) / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(4.4 * ncols, 4.4 * nrows),
                         sharex=True, sharey=True, constrained_layout=True)
axes = np.atleast_1d(axes)

mappables = []
for ax, (name, result) in zip(axes.flat, results.items()):
    _, _, hexes = plot_zphot_zspec_comparison(result['y_true'], result['y_pred'], ax=ax, title=name)
    mappables.append(hexes)

# Blank the leftover slot (5 feature sets in a 2x3 grid).
for ax in list(axes.flat)[len(results):]:
    ax.set_visible(False)

# One color scale shared by every panel. Without this each hexbin autoscales on
# its own counts and the same darkness would mean a different count per panel.
vmax = max(h.get_array().max() for h in mappables if h is not None)
for h in mappables:
    if h is not None:
        h.set_clim(1, vmax)

fig.colorbar(mappables[0], ax=axes.ravel().tolist(),
             label='Galaxies per cell', shrink=0.6)
_ = fig.suptitle('Random Forest photo-zs: feature set comparison', fontsize=14)
# fig.savefig('pz_comparison.png', dpi=200)  # constrained_layout, so no bbox_inches='tight' needed

### 3.d) Adapte código do item **2.b** para criar um painel com todas as métricas comparadas

In [ ]:
SERIES_COLORS = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4']
SERIES_MARKERS = ['o', 's', '^', 'D', 'v']


def plot_binned_metrics(results, bins=None,
                        panels=('bias', 'sigma68', 'out2', 'out3'),
                        bin_by='phot', min_count=25,
                        colors=None, markers=None, ncols=2, figsize=(11, 7)):

    bins = np.asarray(DEFAULT_BINS if bins is None else bins, dtype=float)
    colors = colors if colors is not None else SERIES_COLORS
    markers = markers if markers is not None else SERIES_MARKERS

    nrows = int(np.ceil(len(panels) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, sharex=True,
                             constrained_layout=True)
    axes = np.atleast_1d(axes)

    for i, (name, result) in enumerate(results.items()):
        plot_metrics(result['y_true'], result['y_pred'],
                     bins=bins, panels=panels, bin_by=bin_by, min_count=min_count,
                     axes=axes, label=name,
                     color=colors[i % len(colors)],
                     marker=markers[i % len(markers)],
                     # the first series sets up the panels, the rest only add curves
                     format_axes=(i == 0))

    # Blank any leftover slot (e.g. three metrics on a 2x2 grid)
    for ax in axes.flat[len(panels):]:
        ax.set_visible(False)

    # x label on the bottom-most *visible* panel of each column - with sharex,
    # blanking a slot would otherwise hide the tick labels of the panel above it
    grid = axes if axes.ndim > 1 else axes.reshape(-1, 1)
    xlabel = r'$z_\mathrm{phot}$' if bin_by == 'phot' else r'$z_\mathrm{spec}$'
    for column in grid.T:
        visible = [ax for ax in column if ax.get_visible()]
        if visible:
            visible[-1].set_xlabel(xlabel, fontsize=13)
            visible[-1].tick_params(labelbottom=True)

    axes.flat[0].set_xlim(bins[0], bins[-1])

    handles, labels = axes.flat[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='outside lower center', ncols=len(labels),
               frameon=False, fontsize=10)
    fig.suptitle('Photo-z quality vs redshift, by feature set (bins with < %d '
                 'galaxies dropped)' % min_count, fontsize=13)

    return fig, axes

In [ ]:
fig, axes = plot_binned_metrics(results, bin_by='phot', min_count=100)

# Table view of the same comparison. The legend maps colour+marker to feature set,
# but three of the five hues sit below 3:1 contrast on white, so the numbers must
# also be readable without relying on colour at all.
summary = pd.DataFrame(
    {name: {'MAE':        r['metrics']['mae'],
            'sigma_NMAD': r['metrics']['nmad'],
            'eta [%]':    100 * r['metrics']['eta'],
            'bias':       r['metrics']['bias']}
     for name, r in results.items()}).T
summary.round(4)


## Exercício 4: Construção de distribuições populacionais de redshifts fotométricos

Para as análises cosmológicas que pretendemos fazer, não estamos interessados em seus redshifts individuais, e sim na distribuição populacional dos redshifts, separados por faixas de redshift. O objetivo principal deste exercício é construir estas distribuições e salvá-las para uso posterior.

### 4.a) Construa um histograma da distribuição de redshifts espectroscópicos das galáxias de teste e compare com o histograma do melhor caso dentre os anteriores

In [ ]:
# Line style as a second encoding, for the same reason the panels above carry
# marker shapes: three of the five hues are low contrast on white.
SERIES_LINESTYLES = ['-', '--', '-.', ':', (0, (3, 1, 1, 1))]


def plot_nz(results, bins=None, colors=None, linestyles=None,
            ratio=True, figsize=(10, 7.5)):
    """Compare the true and the recovered redshift distribution N(z).

    The spectroscopic N(z) is drawn once, as a filled grey reference, because it
    is the same test sample for every feature set - five identical curves on top
    of each other would just be ink. Each prediction is a step outline over it.

    Two filled semi-transparent histograms (as in the cell above) are avoided on
    purpose: where they overlap, the eye sees a third colour that belongs to
    neither series and cannot be matched to the legend.

    Args:
        results: dict of {label: {'y_true': ..., 'y_pred': ...}}, as built by the
            training loop. Entries whose true N(z) differs from the first one
            (cross-validation over the full catalogue, say) get their own faint
            reference curve instead of being compared against the wrong sample.

    Kwargs:
        bins: bin edges; defaults to np.arange(0, 1.45, 0.05). Finer bins make
            the ratio panel noisier, not more informative.
        colors, linestyles: per-series styling, cycled if shorter than results.
        ratio: add the predicted/true panel underneath.
        figsize: figure size in inches.

    Returns:
        (fig, axes)
    """
    if bins is None:
        bins = np.arange(0, 1.45, 0.05)
    colors = colors if colors is not None else SERIES_COLORS
    linestyles = linestyles if linestyles is not None else SERIES_LINESTYLES

    centres = 0.5 * (bins[:-1] + bins[1:])

    if ratio:
        fig, axes = plt.subplots(2, 1, figsize=figsize, sharex=True,
                                 gridspec_kw={'height_ratios': [3, 1]},
                                 constrained_layout=True)
        ax_main, ax_ratio = axes
    else:
        fig, ax_main = plt.subplots(figsize=figsize, constrained_layout=True)
        axes, ax_ratio = np.array([ax_main]), None

    reference = None
    for i, (name, result) in enumerate(results.items()):
        y_true = np.asarray(result['y_true'], dtype=float)
        y_pred = np.asarray(result['y_pred'], dtype=float)
        n_true, _ = np.histogram(y_true, bins=bins)
        n_pred, _ = np.histogram(y_pred, bins=bins)

        colour = colors[i % len(colors)]
        style = linestyles[i % len(linestyles)]

        if reference is None:
            reference = n_true
            ax_main.stairs(n_true, bins, fill=True, color='0.87', zorder=1,
                           label='Spectroscopic (truth)')
            ax_main.stairs(n_true, bins, color='0.45', linewidth=1.2, zorder=2)
        elif not np.array_equal(n_true, reference):
            ax_main.stairs(n_true, bins, color=colour, linewidth=1,
                           alpha=0.45, zorder=2)

        ax_main.stairs(n_pred, bins, color=colour, linestyle=style,
                       linewidth=1.8, zorder=3, label=name)

        if ax_ratio is not None:
            # Bins with no true galaxies have no meaningful ratio - leave a gap
            # rather than drawing a spike or a zero.
            with np.errstate(divide='ignore', invalid='ignore'):
                r = np.where(n_true > 0, n_pred/np.where(n_true > 0, n_true, 1), np.nan)
            ax_ratio.plot(centres, r, color=colour, linestyle=style, linewidth=1.6)

    ax_main.set_ylabel('Galaxies per bin', fontsize=12)
    ax_main.set_xlim(bins[0], bins[-1])

    for ax in axes:
        ax.grid(alpha=0.25, linewidth=0.5)
        ax.set_axisbelow(True)
        ax.spines[['top', 'right']].set_visible(False)
        ax.tick_params(labelsize=10)

    if ax_ratio is not None:
        ax_ratio.axhline(1.0, color='0.55', linestyle='--', linewidth=1, zorder=1)
        ax_ratio.set_ylabel('pred / true', fontsize=11)
        ax_ratio.set_xlabel(r'$z$', fontsize=13)
    else:
        ax_main.set_xlabel(r'$z$', fontsize=13)

    handles, labels = ax_main.get_legend_handles_labels()
    fig.legend(handles, labels, loc='outside lower center',
               ncols=3, frameon=False, fontsize=10)
    fig.suptitle('Recovered vs true redshift distribution', fontsize=13)

    return fig, axes


In [ ]:
fig, axes = plot_nz(results)


### 4.b) $N(z)$ suavizado por KDE gaussiano com largura $\sigma_{68}(z)$

Um histograma trata cada photo-z como um valor exato, mas cada photo-z carrega uma incerteza — que já medimos no Exercício 2: $\sigma_{68}(z)$. "Empilhar as PDFs", como se diz na literatura de photo-z, é colocar uma gaussiana centrada no photo-z de cada galáxia e somar — um KDE *adaptativo*, com uma largura por galáxia.

Dois detalhes:

- $\sigma_{68}$ é medido sobre o resíduo normalizado $\Delta z/(1+z)$, então a largura em redshift é $\sigma_z = \sigma_{68}(z)\,(1+z)$;
- `scipy.stats.gaussian_kde` usa **uma única largura** para a amostra inteira (regra de Scott) — desenhamos como comparação. Para uma largura por galáxia, usamos `KDEpy.TreeKDE`, que aceita `bw` vetorial.

In [ ]:
from scipy.stats import gaussian_kde
from KDEpy import TreeKDE

target = 'Colors + mag_i + errors'
z_spec = np.asarray(results[target]['y_true'], dtype=float)
z_phot = np.asarray(results[target]['y_pred'], dtype=float)

# sigma_68 por faixa de redshift (Exercício 2), interpolado no photo-z de cada galáxia
stats = binned_metrics(z_spec, z_phot, bins=np.arange(0, 1.45, 0.05), bin_by='phot', min_count=25)
sigma_z = np.interp(z_phot, stats['z'], stats['sigma68']) * (1 + z_phot)

z_grid = np.linspace(0, 2, 2001)
nz_single = gaussian_kde(z_phot)(z_grid)
nz_adaptive = TreeKDE(kernel='gaussian', bw=sigma_z).fit(z_phot).evaluate(z_grid)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5), constrained_layout=True)

ax.hist(z_spec, bins=np.arange(0, 1.45, 0.05), density=True, color='0.87',
        edgecolor='0.45', label='Espectroscópico (verdade)')
ax.plot(z_grid, nz_single, lw=2, label='KDE scipy, largura única (Scott)')
ax.plot(z_grid, nz_adaptive, lw=2, label=r'KDE adaptativo, $\sigma_{68}(z)\,(1+z)$')

ax.set_xlim(0, 1.4)
ax.set_xlabel(r'$z$', fontsize=13)
ax.set_ylabel(r'$n(z)$  (área unitária)', fontsize=12)
ax.grid(alpha=0.25, linewidth=0.5)
ax.spines[['top', 'right']].set_visible(False)
ax.legend(frameon=False, fontsize=10)
_ = ax.set_title('%s: $N(z)$ suavizado vs verdade' % target, fontsize=13)

### 4.c) $N(z)$ tomográfico: seleção em faixas de photo-z, depois suavização

Para a análise cosmológica, dividimos a amostra em faixas ("bins tomográficos") de $z_\mathrm{phot}$ — o único redshift disponível quando não há espectroscopia — e aplicamos o mesmo KDE adaptativo dentro de cada faixa. Cada curva é normalizada para área unitária na grade exportada, e a tabela salva em CSV usa exatamente a mesma grade das curvas desenhadas.

In [ ]:
tomo_edges = [0.2, 0.4, 0.6, 0.8, 1.0, 1.2]

fig, ax = plt.subplots(figsize=(9, 5.5), constrained_layout=True)

nz_table = {}
for low, high in zip(tomo_edges[:-1], tomo_edges[1:]):
    sel = (z_phot >= low) & (z_phot < high)
    dens = TreeKDE(kernel='gaussian', bw=sigma_z[sel]).fit(z_phot[sel]).evaluate(z_grid)
    dens /= np.trapezoid(dens, z_grid)   # área unitária na grade exportada
    nz_table['zphot_%.1f_%.1f_ntot_%d' % (low, high, sel.sum())] = dens
    ax.plot(z_grid, dens, lw=2,
            label=r'$%.1f \leq z_\mathrm{phot} < %.1f$   ($N = %d$)' % (low, high, sel.sum()))

ax.set_xlim(0, 1.4)
ax.set_ylim(bottom=0)
ax.set_xlabel(r'$z$', fontsize=13)
ax.set_ylabel(r'$n(z)$  (área unitária)', fontsize=12)
ax.grid(alpha=0.25, linewidth=0.5)
ax.spines[['top', 'right']].set_visible(False)
ax.legend(frameon=False, fontsize=9.5)
_ = ax.set_title('$N(z)$ suavizado por bin tomográfico de photo-z', fontsize=13)

In [ ]:
nz_table = pd.DataFrame(nz_table, index=pd.Index(z_grid, name='z'))
nz_table.to_csv('nz_kde_tomographic.csv', float_format='%.10g')
print('salvo: nz_kde_tomographic.csv  (%d linhas x %d colunas)' % nz_table.shape)

# SOBRAS

#### A side note: a practical example of overfitting

In [ ]:
#y_train_pred_rf, y_train = train_and_evaluate_rf(X_train, y_train, X_train, y_train)

### 1.c) Use k-fold cross-validation

In [ ]:
n_folds = 5
del rf
rf = RandomForestRegressor(n_estimators=50,
                           max_depth=30,
                           max_features=None) # FIXME: Which hyperparameters should we use? (n_estimators, max_depth, etc.)


y_pred_cross_val = cross_val_predict(rf, mags, redshifts, cv=n_folds)

In [ ]:
# Results on cross-validation - more realistic!
_ = calculate_metrics(redshifts, y_pred_cross_val)
_ = plot_predictions(redshifts, y_pred_cross_val, "Random Forest Predictions on Cross-Validation")
_ = plot_predictions_density(redshifts, y_pred_cross_val, "Random Forest Predictions on Cross-Validation", cmap='RdYlBu')

## 2. Treino/teste de três algoritmos de aprendizado de máquina

### 2.1. Rede neural (Multi-layer perceptron)

In [ ]:
from sklearn.neural_network import MLPRegressor

mlp = MLPRegressor(random_state=42)
mlp.fit(X_train, y_train)
y_pred_mlp = mlp.predict(X_test)

mlp_test_mae = mean_absolute_error(y_test, y_pred_mlp)
print("Multi-Layer Perceptron Validation MAE:", mlp_test_mae)

### 2.2. Árvore de decisão

In [ ]:
from sklearn.tree import DecisionTreeRegressor

dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

dt_test_mae = mean_absolute_error(y_test, y_pred_dt)
print("Decision Tree Validation MAE:", dt_test_mae)

In [ ]:
# Compare plots from three results

titles = ["Multi-Layer perceptron", "Decision tree", "Random forest"]
data_sets = [y_pred_mlp, y_pred_dt, y_pred_rf]
colors = ["C0", "C1", "C2"]

plt.figure(figsize=(16, 4))

plt.subplot(1, 3, 1)
plt.title("Random Forest")
plt.scatter(y_test, y_pred_rf, s=2, color="C0")
x = np.linspace(0, 1.6, 100)
plt.plot(x, x, "k--", alpha=0.5)
plt.ylim(0, 1.6)
plt.xlim(0, 1.6)
plt.xlabel(r"$z_\mathrm{true}$")
plt.ylabel(r"$z_\mathrm{phot}$")

plt.subplot(1, 3, 2)
plt.title("Decision Tree")
plt.scatter(y_test, y_pred_dt, s=2, color="C1")
x = np.linspace(0, 1.6, 100)
plt.plot(x, x, "k--", alpha=0.5)
plt.ylim(0, 1.6)
plt.xlim(0, 1.6)
plt.xlabel(r"$z_\mathrm{true}$")
plt.ylabel(r"$z_\mathrm{phot}$")

plt.subplot(1, 3, 3)
plt.title("Multi-Layer Perceptron")
plt.scatter(y_test, y_pred_mlp, s=2, color="C2")
x = np.linspace(0, 1.6, 100)
plt.plot(x, x, "k--", alpha=0.5)
plt.ylim(0, 1.6)
plt.xlim(0, 1.6)
_ = plt.xlabel(r"$z_\mathrm{true}$")
_ = plt.ylabel(r"$z_\mathrm{phot}$")